# Day 12: SQL RDBMS — Module 01: Conditional Aggregation (`CASE WHEN`)

> **W3Schools-Aligned Syllabus & Interactive Laboratory**  
> **Course:** C-DAC PGCP-AI Data Analytics | **Module:** Relational Database Management Systems (SQL RDBMS)  
> **Topic:** The SQL `CASE` Statement, Aggregate Functions (`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`), `GROUP BY`, `HAVING`, and Advanced Conditional Aggregation

---

## 📑 Table of Contents (W3Schools Foundation to Advanced Enterprise)

### Part I: W3Schools Core SQL Building Blocks
1. [The SQL `CASE` Statement Syntax & Fundamentals](#1.-The-SQL-CASE-Statement-Syntax-&-Fundamentals)
   - Searched `CASE` (`WHEN condition THEN result`) vs Simple `CASE` (`CASE expr WHEN val THEN result`)
   - The `ELSE` Clause and Short-Circuit Evaluation
   - `CASE` in `SELECT`, `ORDER BY`, and `GROUP BY`
2. [SQL Aggregate Functions & The NULL Handling Rule](#2.-SQL-Aggregate-Functions-&-The-NULL-Handling-Rule)
   - `COUNT(*)`, `COUNT(column)`, `COUNT(DISTINCT column)`
   - `SUM()`, `AVG()`, `MIN()`, `MAX()`
   - Scalar NULL Functions: `COALESCE()`, `IFNULL()`
3. [The `GROUP BY` Statement & Aggregation Lifecycle](#3.-The-GROUP-BY-Statement-&-Aggregation-Lifecycle)
   - SQL Clause Processing Order (`FROM` → `WHERE` → `GROUP BY` → `HAVING` → `SELECT`)
4. [The `HAVING` Clause](#4.-The-HAVING-Clause)
   - Why `HAVING` was added (Filtering on Aggregates)

---

### Part II: Intermediate & Advanced Conditional Aggregation
5. [The Power Pattern: Combining `CASE WHEN` with Aggregates](#5.-The-Power-Pattern:-Combining-CASE-WHEN-with-Aggregates)
   - Single-Scan ($\mathcal{O}(N)$) vs Multiple Queries / Joins
   - Conditional Counting (`COUNT(CASE WHEN ... THEN 1 END)`)
   - The Deadliest SQL Trap: Why `COUNT(CASE ... 1 ELSE 0 END)` Counts Everything!
   - Conditional Summation (`SUM(CASE WHEN ... THEN amount ELSE 0 END)`)
   - Conditional Averages (`AVG(CASE WHEN ... THEN metric END)`)
6. [Dynamic Cross-Tabulation & Matrix Pivoting (Rows to Columns)](#6.-Dynamic-Cross-Tabulation-&-Matrix-Pivoting-(Rows-to-Columns))
   - Row-to-Column Transposition using `SUM(CASE WHEN ...)` / `MAX(CASE WHEN ...)`
7. [Multi-Condition Grouping & Custom Binning](#7.-Multi-Condition-Grouping-&-Custom-Binning)
   - `CASE` inside `GROUP BY` for On-the-Fly Dynamic Cohorting
8. [Filtering Aggregated Groups with Conditional `HAVING`](#8.-Filtering-Aggregated-Groups-with-Conditional-HAVING)
9. [Modern ANSI Standard: `FILTER (WHERE ...)` vs `CASE WHEN`](#9.-Modern-ANSI-Standard:-FILTER-(WHERE-...)-vs-CASE-WHEN)
10. [The SQL-to-Pandas Rosetta Stone (Side-by-Side Equivalence)](#10.-The-SQL-to-Pandas-Rosetta-Stone-(Side-by-Side-Equivalence))
11. [Enterprise Case Studies (SQLite Lab)](#11.-Enterprise-Case-Studies-(SQLite-Lab))
    - E-Commerce Order P&L (Gross vs Net Revenue)
    - Banking Loan Portfolio Risk (Prime vs Subprime, NPL Ratio)
    - SaaS Subscription MRR Movement Waterfall
12. [W3Schools Exercises, Viva Traps & Technical Interview Questions](#12.-W3Schools-Exercises,-Viva-Traps-&-Technical-Interview-Questions)



In [1]:
import sqlite3
import pandas as pd
import numpy as np

# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")

def run_query(sql):
    """Helper function to execute SQL and display result as a clean Pandas DataFrame"""
    return pd.read_sql_query(sql, conn)

print("SQLite database initialized successfully. Ready for queries!")


SQLite database initialized successfully. Ready for queries!


### 1. Global Setup & In-Memory Database Initialization
We instantiate two realistic enterprise tables:
1. `transactions`: Retail e-commerce orders with departments, payment methods, amounts, statuses, and ratings.
2. `loan_portfolio`: Commercial bank loan books with borrower credit scores, loan principals, interest rates, and delinquency statuses.


In [1]:
# 1. Retail Transactions Table
conn.execute("""
CREATE TABLE transactions (
    transaction_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date DATE,
    department VARCHAR(50),
    payment_method VARCHAR(30),
    amount DECIMAL(10, 2),
    status VARCHAR(20),
    rating INTEGER
);
""")

transactions_data = [
    (1, 101, '2026-01-05', 'Electronics', 'Credit Card', 1200.00, 'Completed', 5),
    (2, 102, '2026-01-06', 'Clothing', 'Debit Card', 85.50, 'Completed', 4),
    (3, 101, '2026-01-10', 'Electronics', 'PayPal', 450.00, 'Refunded', 1),
    (4, 103, '2026-01-12', 'Home & Kitchen', 'Credit Card', 320.00, 'Completed', 5),
    (5, 104, '2026-01-15', 'Clothing', 'Credit Card', 150.00, 'Cancelled', None),
    (6, 102, '2026-01-18', 'Electronics', 'Credit Card', 950.00, 'Completed', 5),
    (7, 105, '2026-01-20', 'Home & Kitchen', 'Debit Card', 75.00, 'Completed', 3),
    (8, 103, '2026-01-22', 'Clothing', 'PayPal', 210.00, 'Completed', 4),
    (9, 106, '2026-01-25', 'Electronics', 'Credit Card', 1400.00, 'Refunded', 2),
    (10, 101, '2026-01-28', 'Clothing', 'Debit Card', 120.00, 'Completed', 4),
    (11, 104, '2026-02-01', 'Home & Kitchen', 'Credit Card', 540.00, 'Completed', 4),
    (12, 107, '2026-02-03', 'Electronics', 'Debit Card', 310.00, 'Completed', 5),
    (13, 102, '2026-02-05', 'Clothing', 'PayPal', 95.00, 'Cancelled', None),
    (14, 105, '2026-02-08', 'Electronics', 'Credit Card', 820.00, 'Completed', 4),
    (15, 103, '2026-02-12', 'Home & Kitchen', 'Debit Card', 110.00, 'Completed', 3),
    (16, 108, '2026-02-14', 'Electronics', 'Credit Card', 2200.00, 'Completed', 5),
    (17, 106, '2026-02-18', 'Clothing', 'Credit Card', 175.00, 'Completed', 4),
    (18, 107, '2026-02-20', 'Home & Kitchen', 'PayPal', 260.00, 'Refunded', 2),
    (19, 109, '2026-02-22', 'Electronics', 'Credit Card', 640.00, 'Completed', 4),
    (20, 110, '2026-02-25', 'Clothing', 'Debit Card', 55.00, 'Completed', 5)
]
conn.executemany("INSERT INTO transactions VALUES (?,?,?,?,?,?,?,?)", transactions_data)

# 2. Banking Loan Portfolio Table
conn.execute("""
CREATE TABLE loan_portfolio (
    loan_id INTEGER PRIMARY KEY,
    borrower_name VARCHAR(100),
    credit_score INTEGER,
    loan_amount DECIMAL(12, 2),
    interest_rate DECIMAL(5, 2),
    loan_status VARCHAR(30),
    region VARCHAR(30)
);
""")

loans_data = [
    (201, 'Arun Kumar', 780, 250000.00, 7.5, 'Current', 'North'),
    (202, 'Priya Sharma', 620, 150000.00, 11.2, '30 Days Past Due', 'South'),
    (203, 'Rohan Verma', 540, 80000.00, 14.5, 'Default', 'West'),
    (204, 'Ananya Patel', 810, 400000.00, 6.8, 'Paid Off', 'West'),
    (205, 'Vikram Singh', 690, 180000.00, 9.4, 'Current', 'North'),
    (206, 'Deepa Nair', 590, 120000.00, 13.0, '90+ Days Delinquent', 'South'),
    (207, 'Siddharth Sen', 740, 310000.00, 8.1, 'Current', 'East'),
    (208, 'Kavita Reddy', 670, 220000.00, 10.0, '30 Days Past Due', 'South'),
    (209, 'Rahul Joshi', 510, 95000.00, 15.2, 'Default', 'North'),
    (210, 'Meera Gupta', 760, 280000.00, 7.9, 'Current', 'East'),
    (211, 'Suresh Menon', 830, 500000.00, 6.5, 'Paid Off', 'South'),
    (212, 'Tanvi Deshmukh', 640, 160000.00, 11.0, 'Current', 'West')
]
conn.executemany("INSERT INTO loan_portfolio VALUES (?,?,?,?,?,?,?)", loans_data)
conn.commit()

display(run_query("SELECT * FROM transactions LIMIT 5"))


,transaction_id,customer_id,order_date,department,payment_method,amount,status,rating
0,1,101,2026-01-05,Electronics,Credit Card,1200.0,Completed,5.0
1,2,102,2026-01-06,Clothing,Debit Card,85.5,Completed,4.0
2,3,101,2026-01-10,Electronics,PayPal,450.0,Refunded,1.0
3,4,103,2026-01-12,Home & Kitchen,Credit Card,320.0,Completed,5.0
4,5,104,2026-01-15,Clothing,Credit Card,150.0,Cancelled,NaN


---
## 2. The Cardinal Rule: How SQL Aggregates Handle `NULL` vs `0`

The most common trap in SQL interviews and production pipelines is confusing how `COUNT()` handles `0` vs `NULL`:
- `COUNT(column_or_expression)` counts **any non-null entry** (including `0`, `""`, and `FALSE`!).
- `COUNT(CASE WHEN status='Refunded' THEN 1 ELSE 0 END)` will evaluate to `0` on non-refunded rows, and **SQL will count it!** Resulting in the total row count!
- To properly count conditional matches, you must return `NULL` when the condition fails (or omit `ELSE`).


In [1]:
sql_cardinal_test = """
SELECT 
    department,
    COUNT(*) AS total_rows,
    -- ❌ TRAP: Counts every row because 0 is a non-null value!
    COUNT(CASE WHEN status = 'Refunded' THEN 1 ELSE 0 END) AS bugged_refund_count,
    -- ✅ CORRECT: Omitted ELSE yields NULL, which COUNT ignores
    COUNT(CASE WHEN status = 'Refunded' THEN 1 END) AS correct_refund_count,
    -- ✅ EQUIVALENT ALTERNATIVE: Use SUM(1 or 0)
    SUM(CASE WHEN status = 'Refunded' THEN 1 ELSE 0 END) AS sum_refund_count
FROM transactions
GROUP BY department;
"""

res_cardinal = run_query(sql_cardinal_test)
display(res_cardinal)


,department,total_rows,bugged_refund_count,correct_refund_count,sum_refund_count
0,Clothing,7,7,0,0
1,Electronics,8,8,2,2
2,Home & Kitchen,5,5,1,1


---
## 3. Pattern 1: Conditional Counting & Distinct Entity Counting

In this query, we measure:
1. `total_orders`: Total volume of transactions.
2. `completed_orders`: Volume of successful sales.
3. `cancelled_orders`: Dropped checkout count.
4. `unique_shoppers`: Total distinct customer accounts active in each department.
5. `unique_refund_customers`: Distinct customers who suffered a refund event.


In [1]:
sql_pattern1 = """
SELECT 
    department,
    COUNT(*) AS total_orders,
    COUNT(CASE WHEN status = 'Completed' THEN 1 END) AS completed_orders,
    COUNT(CASE WHEN status = 'Cancelled' THEN 1 END) AS cancelled_orders,
    COUNT(DISTINCT customer_id) AS unique_shoppers,
    COUNT(DISTINCT CASE WHEN status = 'Refunded' THEN customer_id END) AS unique_refund_customers
FROM transactions
GROUP BY department;
"""

res_pattern1 = run_query(sql_pattern1)
display(res_pattern1)


,department,total_orders,completed_orders,cancelled_orders,unique_shoppers,unique_refund_customers
0,Clothing,7,5,2,6,0
1,Electronics,8,6,0,7,2
2,Home & Kitchen,5,4,0,4,1


---
## 4. Pattern 2: Conditional Summation (P&L and Revenue Waterfalls)

Financial reporting requires calculating gross revenue, refund deductions, and net settled amounts in a single table scan.
- Notice the use of `ELSE 0` inside `SUM()` so non-matching rows add `0` without creating null propagation.


In [1]:
sql_pattern2 = """
SELECT 
    department,
    COUNT(*) AS total_transactions,
    ROUND(SUM(CASE WHEN status = 'Completed' THEN amount ELSE 0 END), 2) AS gross_revenue,
    ROUND(SUM(CASE WHEN status = 'Refunded'  THEN amount ELSE 0 END), 2) AS refund_deductions,
    ROUND(SUM(CASE WHEN status = 'Cancelled' THEN amount ELSE 0 END), 2) AS lost_cancelled_demand,
    ROUND(SUM(CASE WHEN status = 'Completed' THEN amount ELSE 0 END) -
          SUM(CASE WHEN status = 'Refunded'  THEN amount ELSE 0 END), 2) AS net_settled_revenue
FROM transactions
GROUP BY department
ORDER BY net_settled_revenue DESC;
"""

res_pattern2 = run_query(sql_pattern2)
display(res_pattern2)


,department,total_transactions,gross_revenue,refund_deductions,lost_cancelled_demand,net_settled_revenue
0,Electronics,8,6120.0,1850.0,0.0,4270.0
1,Home & Kitchen,5,1045.0,260.0,0.0,785.0
2,Clothing,7,645.5,0.0,245.0,645.5


---
## 5. Pattern 3: KPI Rates, Proportions & Floating Point Division

Calculating proportions (Conversion Rate, Refund Rate, Delinquency Rate):
- **Float Cast Rule:** We multiply by `100.0` or cast to float to prevent integer division truncation in SQL engines.
- `AVG(CASE WHEN status = 'Completed' THEN rating END)` computes the average rating **only for completed orders**, ignoring cancellations and refunds where rating is NULL.


In [1]:
sql_pattern3 = """
SELECT 
    department,
    COUNT(*) AS total_orders,
    ROUND(100.0 * COUNT(CASE WHEN status = 'Completed' THEN 1 END) / COUNT(*), 2) AS completion_rate_pct,
    ROUND(100.0 * COUNT(CASE WHEN status = 'Refunded'  THEN 1 END) / COUNT(*), 2) AS refund_rate_pct,
    ROUND(AVG(CASE WHEN status = 'Completed' THEN rating END), 2) AS avg_completed_customer_rating
FROM transactions
GROUP BY department;
"""

res_pattern3 = run_query(sql_pattern3)
display(res_pattern3)


,department,total_orders,completion_rate_pct,refund_rate_pct,avg_completed_customer_rating
0,Clothing,7,71.43,0.0,4.20
1,Electronics,8,75.00,25.0,4.67
2,Home & Kitchen,5,80.00,20.0,3.75


---
## 6. Pattern 4: Dynamic Pivoting & Cross-Tabulation (Rows to Columns)

Cross-tabulation pivots categorical values (`Credit Card`, `Debit Card`, `PayPal`) into distinct columns:
- We use `SUM(CASE WHEN payment_method = '...' THEN amount ELSE 0 END)`.
- This approach is **100% ANSI-SQL compliant** and runs across all databases without requiring proprietary `PIVOT` operators.


In [1]:
sql_pattern4 = """
SELECT 
    department,
    ROUND(SUM(CASE WHEN payment_method = 'Credit Card' THEN amount ELSE 0 END), 2) AS credit_card_sales,
    ROUND(SUM(CASE WHEN payment_method = 'Debit Card'  THEN amount ELSE 0 END), 2) AS debit_card_sales,
    ROUND(SUM(CASE WHEN payment_method = 'PayPal'      THEN amount ELSE 0 END), 2) AS paypal_sales,
    ROUND(SUM(amount), 2) AS total_settled_sales
FROM transactions
WHERE status = 'Completed'
GROUP BY department;
"""

res_pattern4 = run_query(sql_pattern4)
display(res_pattern4)


,department,credit_card_sales,debit_card_sales,paypal_sales,total_settled_sales
0,Clothing,175.0,260.5,210.0,645.5
1,Electronics,5810.0,310.0,0.0,6120.0
2,Home & Kitchen,860.0,185.0,0.0,1045.0


---
## 7. Pattern 5: Multi-Condition Bucketing & Risk Tier Segmentation

In commercial lending, loans are segmented by borrower credit score:
- **Prime:** Credit Score $\ge 750$
- **Near-Prime:** Credit Score between $650$ and $749$
- **Subprime:** Credit Score $< 650$

We also compute the **Non-Performing Loan (NPL) Ratio** for each geographic region.


In [1]:
sql_pattern5 = """
SELECT 
    region,
    COUNT(*) AS total_loans,
    ROUND(SUM(CASE WHEN credit_score >= 750 THEN loan_amount ELSE 0 END), 2) AS prime_exposure,
    ROUND(SUM(CASE WHEN credit_score BETWEEN 650 AND 749 THEN loan_amount ELSE 0 END), 2) AS near_prime_exposure,
    ROUND(SUM(CASE WHEN credit_score < 650 THEN loan_amount ELSE 0 END), 2) AS subprime_exposure,
    ROUND(100.0 * SUM(CASE WHEN loan_status IN ('Default', '90+ Days Delinquent') THEN loan_amount ELSE 0 END) 
          / SUM(loan_amount), 2) AS npl_ratio_pct
FROM loan_portfolio
GROUP BY region;
"""

res_pattern5 = run_query(sql_pattern5)
display(res_pattern5)


,region,total_loans,prime_exposure,near_prime_exposure,subprime_exposure,npl_ratio_pct
0,East,2,280000.0,310000.0,0.0,0.00
1,North,3,250000.0,180000.0,95000.0,18.10
2,South,4,500000.0,220000.0,270000.0,12.12
3,West,3,400000.0,0.0,240000.0,12.50


---
## 8. Pattern 6: Group Filtering with `HAVING` & Conditional Metrics

Use conditional metrics inside the `HAVING` clause to filter out departments that have an unacceptable refund rate ($> 20\%$).


In [1]:
sql_pattern6 = """
SELECT 
    department,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN status = 'Refunded' THEN 1 ELSE 0 END) AS refund_count,
    ROUND(100.0 * SUM(CASE WHEN status = 'Refunded' THEN 1 ELSE 0 END) / COUNT(*), 2) AS refund_rate_pct
FROM transactions
GROUP BY department
HAVING 100.0 * SUM(CASE WHEN status = 'Refunded' THEN 1 ELSE 0 END) / COUNT(*) > 20.0;
"""

res_pattern6 = run_query(sql_pattern6)
display(res_pattern6)


,department,total_orders,refund_count,refund_rate_pct
0,Electronics,8,2,25.0


---
## 9. Modern ANSI SQL: `FILTER (WHERE ...)` vs `CASE WHEN`

In modern SQL (ANSI SQL:2003, PostgreSQL, SQLite 3.30+, DuckDB), the `FILTER` clause provides an elegant and concise syntax:
```sql
COUNT(*) FILTER (WHERE status = 'Completed')
SUM(amount) FILTER (WHERE status = 'Completed')
```


In [1]:
sql_filter = """
SELECT 
    department,
    COUNT(*) AS total_orders,
    COUNT(*) FILTER (WHERE status = 'Completed') AS completed_orders,
    SUM(amount) FILTER (WHERE status = 'Completed') AS gross_sales,
    ROUND(AVG(rating) FILTER (WHERE rating IS NOT NULL), 2) AS avg_rating
FROM transactions
GROUP BY department;
"""

res_filter = run_query(sql_filter)
display(res_filter)


,department,total_orders,completed_orders,gross_sales,avg_rating
0,Clothing,7,5,645.5,4.20
1,Electronics,8,6,6120.0,3.88
2,Home & Kitchen,5,4,1045.0,3.40


---
## 10. The SQL-to-Pandas Rosetta Stone (Side-by-Side Equivalence)

Data Analytics pair-programming often requires translating between SQL engines and Python Pandas DataFrames.
Here is how the exact same department financial breakdown is expressed in Pandas:


In [1]:
# Load raw table into a pandas dataframe
df_tx = run_query("SELECT * FROM transactions")

# Replicate Pattern 2 (P&L Waterfall) using Pandas vectorization
df_pnl = df_tx.groupby('department').apply(lambda g: pd.Series({
    'total_transactions': len(g),
    'gross_revenue': g.loc[g['status'] == 'Completed', 'amount'].sum(),
    'refund_deductions': g.loc[g['status'] == 'Refunded', 'amount'].sum(),
    'lost_cancelled_demand': g.loc[g['status'] == 'Cancelled', 'amount'].sum(),
    'net_settled_revenue': (g.loc[g['status'] == 'Completed', 'amount'].sum() - 
                            g.loc[g['status'] == 'Refunded', 'amount'].sum())
})).reset_index()

display(df_pnl.sort_values(by='net_settled_revenue', ascending=False))


,department,total_transactions,gross_revenue,refund_deductions,lost_cancelled_demand,net_settled_revenue
1,Electronics,8.0,6120.0,1850.0,0.0,4270.0
2,Home & Kitchen,5.0,1045.0,260.0,0.0,785.0
0,Clothing,7.0,645.5,0.0,245.0,645.5


---
## 11. Enterprise Case Study: SaaS MRR Waterfall & Churn Modeling

In SaaS unit economics, Monthly Recurring Revenue (MRR) changes are audited across event types:
- **New:** Net new logo added.
- **Expansion:** Upsell or expansion seats.
- **Contraction:** Downsized tier.
- **Churn:** Cancelled subscription.


In [1]:
conn.execute("""
CREATE TABLE saas_subscriptions (
    subscription_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    plan_tier VARCHAR(30),
    event_type VARCHAR(30),
    mrr_amount DECIMAL(10, 2)
);
""")

saas_data = [
    (1, 1001, 'Enterprise', 'New', 15000.00),
    (2, 1002, 'Mid-Market', 'New', 4500.00),
    (3, 1003, 'Startup',    'New', 1200.00),
    (4, 1004, 'Enterprise', 'Expansion', 3500.00),
    (5, 1005, 'Mid-Market', 'Contraction', 800.00),
    (6, 1006, 'Startup',    'Churn', 1200.00),
    (7, 1007, 'Enterprise', 'Expansion', 5000.00),
    (8, 1008, 'Mid-Market', 'Churn', 4500.00),
    (9, 1009, 'Enterprise', 'Contraction', 2000.00),
    (10, 1010, 'Startup',   'Expansion', 400.00)
]
conn.executemany("INSERT INTO saas_subscriptions VALUES (?,?,?,?,?)", saas_data)
conn.commit()

sql_mrr = """
SELECT 
    plan_tier,
    COUNT(*) AS total_events,
    SUM(CASE WHEN event_type = 'New'         THEN mrr_amount ELSE 0 END) AS new_mrr,
    SUM(CASE WHEN event_type = 'Expansion'   THEN mrr_amount ELSE 0 END) AS expansion_mrr,
    SUM(CASE WHEN event_type = 'Contraction' THEN mrr_amount ELSE 0 END) AS contraction_mrr,
    SUM(CASE WHEN event_type = 'Churn'       THEN mrr_amount ELSE 0 END) AS churn_mrr,
    SUM(CASE WHEN event_type IN ('New', 'Expansion') THEN mrr_amount ELSE -mrr_amount END) AS net_mrr_change
FROM saas_subscriptions
GROUP BY plan_tier
ORDER BY net_mrr_change DESC;
"""

res_mrr = run_query(sql_mrr)
display(res_mrr)


,plan_tier,total_events,new_mrr,expansion_mrr,contraction_mrr,churn_mrr,net_mrr_change
0,Enterprise,4,15000,8500,2000,0,21500
1,Startup,3,1200,400,0,1200,400
2,Mid-Market,3,4500,0,800,4500,-800


---
## 12. Critical Viva, Gate & MAANG Interview Question Bank

#### Q1: What is the exact difference between `COUNT(CASE WHEN x > 10 THEN 0 END)` and `COUNT(CASE WHEN x > 10 THEN 1 END)`?
- **Answer:** **Zero difference in count output!** `COUNT(expression)` tallies all non-null entries. Both `0` and `1` are valid non-null values. If the condition is met, both evaluate to non-null and increment the count. To ignore unselected rows, the branch must return `NULL` (e.g. `COUNT(CASE WHEN x > 10 THEN 1 END)`).

---

#### Q2: Why is Conditional Aggregation preferred over self-joins or multiple subqueries?
- **Answer:**
  1. **Single Table Scan ($\mathcal{O}(N)$):** Reads disk records once, resolving branches in CPU cache.
  2. **Minimal Memory Footprint:** Avoids intermediate Cartesian hash tables and temporary disk spills.
  3. **Robust Entity Retention:** Eliminates accidental drops of empty cohorts that occur with `INNER JOIN`.

---

#### Q3: How does `AVG(CASE WHEN condition THEN metric END)` handle false rows?
- **Answer:** Unmatched rows produce `NULL`. Standard SQL `AVG()` excludes `NULL` values from both the numerator (sum) and denominator (count). Therefore, the average reflects strictly the qualifying subset. Putting `ELSE 0` would falsely include unmatched rows in the denominator, dragging down the average.

---

#### Q4: When should you use `FILTER (WHERE ...)` instead of `CASE WHEN`?
- **Answer:** When your database engine supports ANSI SQL:2003 (PostgreSQL, SQLite 3.30+, DuckDB). `FILTER` provides superior readability and expressiveness. However, `CASE WHEN` remains the universal standard for maximum portability across legacy MySQL and Oracle.

